# nb_bronze_to_silver
Bronze Lakehouse の手動 CSV と Fabric SQL DB ミラーを Silver Delta テーブルへ正規化します。再実行時は overwriteSchema で同名テーブルを更新します。


## 前提
Notebook を `lh_nexus6_bronze` と `lh_nexus6_silver` にアタッチし、SQL DB ミラー `sqldb_common_01` / `sqldb_mobile_01..05` / `sqldb_ecommerce_01..05` / `sqldb_fintech_01..05` を Bronze から参照できる状態にします。


In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType, StringType, BooleanType, DateType, TimestampType

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

WORKSPACE_NAME = "fabric_seworkshop_ws1"
BRONZE_LAKEHOUSE_NAME = "lh_nexus6_bronze"
BRONZE_SEED_PATH = f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{BRONZE_LAKEHOUSE_NAME}.Lakehouse/Files/manual_seed/scenario_seed.csv"
SQL_SERVER = "mnlvphuj2ouevmfjuzf2jtaqfe-yktncsoas2ru3io7senrtixqxq.database.fabric.microsoft.com"
SQL_DATABASES = {"sqldb_common_01": "sqldb_common_01-31ed4242-3749-4b67-a4cb-02f2ec32df34", "sqldb_mobile_01": "sqldb_mobile_01-6c0e0c01-8fcf-4e98-b311-190035a69911", "sqldb_ecommerce_01": "sqldb_ecommerce_01-670f0bcb-46e5-4c16-8836-330fd9b73829", "sqldb_fintech_01": "sqldb_fintech_01-529ae3ef-81d8-4fb2-96ec-d0d2f856e040"}

def get_sql_token():
    try:
        return mssparkutils.credentials.getToken("https://database.windows.net/")
    except NameError:
        from notebookutils import mssparkutils as nb_mssparkutils
        return nb_mssparkutils.credentials.getToken("https://database.windows.net/")

SQL_ACCESS_TOKEN = get_sql_token()

def read_jdbc(source_db, table):
    resolved_db = SQL_DATABASES.get(source_db, SQL_DATABASES.get(DOMAIN_FALLBACKS.get(source_db, source_db), source_db))
    url = f"jdbc:sqlserver://{SQL_SERVER}:1433;database={resolved_db};encrypt=true;trustServerCertificate=true;loginTimeout=60"
    props = {"driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver", "accessToken": SQL_ACCESS_TOKEN}
    return spark.read.jdbc(url=url, table=f"dbo.{table}", properties=props)
for schema in ["common", "mobile", "ecommerce", "fintech"]:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema}")

FX_TO_JPY = {"JPY": 1.0, "USD": 160.0, "EUR": 172.0, "CNY": 22.0, "GBP": 202.0}
rate_expr = F.create_map([x for kv in FX_TO_JPY.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

def read_table_any(candidates):
    last_error = None
    for name in candidates:
        try:
            return spark.read.table(name)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"No readable source table found. Tried: {candidates}. Last error: {last_error}")

DOMAIN_FALLBACKS = {
    "sqldb_mobile_02": "sqldb_mobile_01",
    "sqldb_mobile_03": "sqldb_mobile_01",
    "sqldb_mobile_04": "sqldb_mobile_01",
    "sqldb_mobile_05": "sqldb_mobile_01",
    "sqldb_ecommerce_02": "sqldb_ecommerce_01",
    "sqldb_ecommerce_03": "sqldb_ecommerce_01",
    "sqldb_ecommerce_04": "sqldb_ecommerce_01",
    "sqldb_ecommerce_05": "sqldb_ecommerce_01",
    "sqldb_fintech_02": "sqldb_fintech_01",
    "sqldb_fintech_03": "sqldb_fintech_01",
    "sqldb_fintech_04": "sqldb_fintech_01",
    "sqldb_fintech_05": "sqldb_fintech_01",
}

def raw(db, table):
    dbs = [db]
    if db in DOMAIN_FALLBACKS:
        dbs.append(DOMAIN_FALLBACKS[db])
    candidates = []
    for source_db in dbs:
        candidates.extend([
            f"{source_db}.dbo.{table}",
            f"`{source_db}`.`dbo`.`{table}`",
            f"{source_db}.{table}",
            f"`{source_db}`.`{table}`",
        ])
    candidates.append(table)
    try:
        return read_table_any(candidates)
    except Exception as mirror_error:
        last_error = mirror_error
        for source_db in dbs:
            try:
                return read_jdbc(source_db, table)
            except Exception as jdbc_error:
                last_error = jdbc_error
        raise RuntimeError(f"No readable source table found for {db}.{table}. Tried mirror {candidates} and JDBC fallbacks {dbs}. Last error: {last_error}")

def with_ingest(df, source):
    return (df.withColumn("_ingest_date", F.current_date())
              .withColumn("_source_system", F.lit(source)))

def add_missing(df, defaults):
    for name, value in defaults.items():
        if name not in df.columns:
            df = df.withColumn(name, value)
    return df

def dedupe(df, keys, order_col=None):
    if not keys:
        return df.dropDuplicates()
    if order_col and order_col in df.columns:
        w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc_nulls_last())
        return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
    return df.dropDuplicates(keys)

def write_table(df, table, partition_col=None):
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_col and partition_col in df.columns:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(table)

def ym_from(col):
    return F.date_format(F.to_date(F.col(col)), "yyyy-MM")


## 変換
型変換、重複排除、JPY 換算、`year_month` / `_ingest_date` 付与を行い、Silver の `common` / `mobile` / `ecommerce` / `fintech` スキーマへ保存します。


In [ ]:
# Manual representative seed CSV. Upload to Bronze Files/manual_seed/scenario_seed.csv before running.
seed = (spark.read.option("header", True).option("inferSchema", True)
        .csv(BRONZE_SEED_PATH))
write_table(with_ingest(seed, "manual_seed"), "common.scenario_seed")

# Common
unified = add_missing(raw("sqldb_common_01", "unified_customers"), {
    "is_deleted": F.lit(False),
})
unified = unified.withColumn("birth_date", F.to_date("birth_date")) \
                 .withColumn("registered_at", F.to_timestamp("registered_at")) \
                 .withColumn("last_updated_at", F.to_timestamp("last_updated_at"))
write_table(with_ingest(dedupe(unified, ["unified_customer_id"], "last_updated_at"), "sqldb_common_01"), "common.unified_customers")
write_table(with_ingest(raw("sqldb_common_01", "domain_id_mappings").withColumn("linked_at", F.to_timestamp("linked_at")), "sqldb_common_01"), "common.domain_id_mappings")
write_table(with_ingest(raw("sqldb_common_01", "customer_segment_assignments").withColumn("assigned_at", F.to_timestamp("assigned_at")).withColumn("expires_at", F.to_timestamp("expires_at")), "sqldb_common_01"), "common.customer_segments")

# Mobile
contracts = raw("sqldb_mobile_02", "mobile_contracts")
contracts = add_missing(contracts, {
    "contract_start_date": F.to_date(F.lit(None)),
    "contract_end_date": F.to_date(F.lit(None)),
    "status": F.lit("active"),
    "is_deleted": F.lit(False),
    "year_month": F.col("update_month"),
})
write_table(with_ingest(dedupe(contracts, ["contract_id"]), "sqldb_mobile_02"), "mobile.contracts", "year_month")

usage = raw("sqldb_mobile_03", "mobile_usage_billings")
usage = (usage.withColumnRenamed("voice_usage", "voice_usage_min")
              .withColumnRenamed("data_usage", "data_usage_gb")
              .withColumnRenamed("monthly_charge", "monthly_charge_jpy")
              .withColumn("usage_date", F.to_date("usage_date"))
              .withColumn("overage_charge_jpy", F.lit(0).cast(DecimalType(12,2)))
              .withColumn("year_month", ym_from("usage_date")))
write_table(with_ingest(dedupe(usage, ["usage_id"]), "sqldb_mobile_03"), "mobile.usage_billing", "year_month")

costs = raw("sqldb_mobile_04", "mobile_cost_items")
costs = (costs.withColumn("procurement_date", F.to_date("procurement_date"))
              .withColumn("fx_rate_used", F.coalesce(rate_expr[F.col("currency")], F.lit(1.0)).cast(DecimalType(10,4)))
              .withColumn("unit_cost_jpy", (F.col("unit_cost") * F.col("fx_rate_used")).cast(DecimalType(14,2)))
              .withColumn("year_month", ym_from("procurement_date")))
write_table(with_ingest(dedupe(costs, ["cost_item_id"]), "sqldb_mobile_04"), "mobile.device_costs", "year_month")

mnp = raw("sqldb_mobile_02", "mnp_history").withColumn("executed_at", F.to_timestamp("executed_at")).withColumn("year_month", F.date_format("executed_at", "yyyy-MM"))
write_table(with_ingest(dedupe(mnp, ["mnp_id"], "executed_at"), "sqldb_mobile_02"), "mobile.mnp_history", "year_month")
inst = raw("sqldb_mobile_03", "mobile_installments").withColumnRenamed("total_amount", "total_amount_jpy").withColumnRenamed("monthly_payment", "monthly_payment_jpy").withColumn("start_date", F.to_date("start_date")).withColumn("year_month", ym_from("start_date"))
write_table(with_ingest(dedupe(inst, ["installment_id"]), "sqldb_mobile_03"), "mobile.installment_details", "year_month")
tickets = raw("sqldb_mobile_05", "mobile_tickets").withColumn("created_at", F.to_timestamp("created_at")).withColumn("resolved_at", F.to_timestamp("resolved_at")).withColumn("resolution_days", F.datediff("resolved_at", "created_at")).withColumn("year_month", F.date_format("created_at", "yyyy-MM"))
write_table(with_ingest(dedupe(tickets, ["ticket_id"], "resolved_at"), "sqldb_mobile_05"), "mobile.crm_tickets", "year_month")

# Ecommerce
products = raw("sqldb_ecommerce_01", "products").select("sku", "procurement_currency", "cost_price", "selling_price", "category")
write_table(with_ingest(raw("sqldb_ecommerce_04", "members").withColumn("registered_at", F.to_date("registered_at")).withColumn("is_deleted", F.lit(False)), "sqldb_ecommerce_04"), "ecommerce.members")
orders = raw("sqldb_ecommerce_02", "orders").join(products, "sku", "left")
orders = (orders.withColumn("order_date", F.to_date("order_date"))
              .withColumnRenamed("order_amount", "order_amount_jpy")
              .withColumn("fx_rate_used", F.coalesce(rate_expr[F.col("procurement_currency")], F.lit(1.0)).cast(DecimalType(10,4)))
              .withColumn("cost_price_jpy", (F.col("cost_price") * F.col("quantity") * F.col("fx_rate_used")).cast(DecimalType(12,2)))
              .withColumn("gross_margin_jpy", (F.col("order_amount_jpy") - F.col("cost_price_jpy")).cast(DecimalType(12,2)))
              .withColumn("year_month", ym_from("order_date")))
write_table(with_ingest(dedupe(orders, ["order_id"]), "sqldb_ecommerce_02"), "ecommerce.orders", "year_month")
inv = raw("sqldb_ecommerce_03", "inventory")
inv = (inv.withColumn("arrival_date", F.to_date("arrival_date"))
          .withColumn("snapshot_date", F.current_date())
          .withColumn("fx_rate_used", F.coalesce(rate_expr[F.col("import_currency")], F.lit(1.0)))
          .withColumn("import_cost_jpy", (F.col("stock_qty") * F.col("fx_rate_used")).cast(DecimalType(12,2)))
          .withColumn("year_month", ym_from("snapshot_date")))
write_table(with_ingest(dedupe(inv, ["inventory_id"]), "sqldb_ecommerce_03"), "ecommerce.inventory", "year_month")
campaigns = raw("sqldb_ecommerce_05", "point_campaigns").select("campaign_id", "point_rate", "campaign_budget")
reactions = raw("sqldb_ecommerce_05", "campaign_reactions").join(campaigns, "campaign_id", "left")
reactions = reactions.withColumn("reaction_time", F.to_timestamp("reaction_time")).withColumn("year_month", F.date_format("reaction_time", "yyyy-MM"))
write_table(with_ingest(dedupe(reactions, ["reaction_id"], "reaction_time"), "sqldb_ecommerce_05"), "ecommerce.campaign_reactions", "year_month")
write_table(with_ingest(raw("sqldb_ecommerce_04", "point_events").withColumn("event_at", F.to_timestamp("event_at")).withColumn("expiry_at", F.to_timestamp("expiry_at")).withColumn("year_month", F.date_format("event_at", "yyyy-MM")), "sqldb_ecommerce_04"), "ecommerce.point_events", "year_month")
write_table(with_ingest(raw("sqldb_ecommerce_05", "member_behaviors").withColumn("event_time", F.to_timestamp("event_time")).withColumn("year_month", F.date_format("event_time", "yyyy-MM")), "sqldb_ecommerce_05"), "ecommerce.member_behaviors", "year_month")

# Fintech
accounts = raw("sqldb_fintech_01", "accounts").withColumnRenamed("balance", "balance_jpy").withColumn("opened_at", F.to_date("opened_at")).withColumn("balance_currency", F.lit("JPY")).withColumn("balance_original", F.col("balance_jpy")).withColumn("kyc_status", F.lit("verified")).withColumn("is_deleted", F.lit(False))
write_table(with_ingest(dedupe(accounts, ["account_id"]), "sqldb_fintech_01"), "fintech.accounts")
card = raw("sqldb_fintech_02", "card_transactions").withColumnRenamed("transaction_date", "transaction_at").withColumnRenamed("amount", "amount_jpy")
card = card.withColumn("transaction_at", F.to_timestamp("transaction_at")).withColumn("amount_original", F.col("amount_jpy")).withColumn("transaction_currency", F.when(F.col("overseas_flag"), "USD").otherwise("JPY")).withColumn("fx_rate_used", F.coalesce(rate_expr[F.col("transaction_currency")], F.lit(1.0))).withColumn("merchant_id", F.lit(None).cast(StringType())).withColumn("year_month", F.date_format("transaction_at", "yyyy-MM"))
write_table(with_ingest(dedupe(card, ["transaction_id"], "transaction_at"), "sqldb_fintech_02"), "fintech.card_transactions", "year_month")
fxp = raw("sqldb_fintech_04", "fx_positions").withColumnRenamed("pnl_amount", "pnl_jpy").withColumn("position_currency", F.col("market_currency")).withColumn("snapshot_date", F.current_date()).withColumn("year_month", ym_from("snapshot_date"))
write_table(with_ingest(dedupe(fxp, ["position_id"]), "sqldb_fintech_04"), "fintech.fx_positions", "year_month")
loans = raw("sqldb_fintech_05", "loan_balances").withColumnRenamed("principal_balance", "principal_balance_jpy").withColumnRenamed("monthly_payment", "monthly_payment_jpy").withColumn("maturity_date", F.to_date("maturity_date")).withColumn("snapshot_date", F.current_date()).withColumn("year_month", ym_from("snapshot_date"))
write_table(with_ingest(dedupe(loans, ["loan_id"]), "sqldb_fintech_05"), "fintech.loan_balances", "year_month")
rates = raw("sqldb_fintech_04", "fx_rate_snapshots").withColumn("captured_at", F.to_timestamp("captured_at")).withColumn("year_month", F.date_format("captured_at", "yyyy-MM"))
write_table(with_ingest(dedupe(rates, ["rate_snapshot_id"], "captured_at"), "sqldb_fintech_04"), "fintech.fx_rate_snapshots", "year_month")
reviews = raw("sqldb_fintech_05", "credit_reviews").withColumn("reviewed_at", F.to_timestamp("reviewed_at")).withColumn("year_month", F.date_format("reviewed_at", "yyyy-MM"))
write_table(with_ingest(dedupe(reviews, ["review_id"], "reviewed_at"), "sqldb_fintech_05"), "fintech.credit_reviews", "year_month")
write_table(with_ingest(raw("sqldb_fintech_05", "revenue_risks"), "sqldb_fintech_05"), "fintech.revenue_risks")
